# StagHunt checkpoint evaluation

Evaluates StagHunt checkpoints from `not_for_submit/checkpoints_for_eval_SH_8e`. The default target is the noisy-observation scenario (`sh_noise`), and the metric is `success_rate`.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

MPLCONFIGDIR = Path("/tmp") / "pci_marl_matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "main.py").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root.")


ROOT = resolve_repo_root()
FIGURES_DIR = ROOT / "figures"
CHECKPOINTS_DIR = ROOT / "not_for_submit" / "checkpoints_for_eval_SH_8e"

for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sh_lbf_checkpoint_eval_utils as clu
clu = importlib.reload(clu)

DEVICE = "cpu"
CHECKPOINT_CHOICE = "final"  # or "best"
DUPLICATE_POLICY = "latest"
EVAL_SEEDS = list(range(10001, 10065))
EVAL_EPISODES = 1
EVAL_SEED_BATCH_SIZE = 16
TARGET_SCENARIOS = ["sh"]
SCENARIO_RESULTS = {}

clu.bootstrap_project(DEVICE)
runs_df = clu.discover_staghunt_runs(CHECKPOINTS_DIR, scenario_filter=TARGET_SCENARIOS)
if runs_df.empty:
    raise RuntimeError(f"No StagHunt runs were discovered under {CHECKPOINTS_DIR}")

In [ ]:
TARGET_MODELS = clu.ordered_model_labels(runs_df["model_label"].dropna().unique().tolist(), clu.SH_MODEL_ORDER)

print(f"Repo root: {ROOT}")
print(f"Loaded {len(runs_df)} StagHunt evaluation runs from {CHECKPOINTS_DIR}")
print(f"Scenarios: {TARGET_SCENARIOS}")
print(f"Models: {TARGET_MODELS}")
display(runs_df.groupby(["scenario", "model_label"]).size().unstack(fill_value=0))
display(runs_df[["scenario", "model_label", "seed", "wandb_run_name", "final_checkpoint"]].head())

In [ ]:
# Smoke test: one training seed x two evaluation seeds.
SMOKE_EVAL_SEEDS = EVAL_SEEDS[:2]
smoke_row = runs_df.sort_values(["scenario", "model_label", "seed", "run_dir"]).iloc[0]
smoke_runs = runs_df[
    (runs_df["scenario"] == smoke_row["scenario"])
    & (runs_df["model_label"] == smoke_row["model_label"])
    & (runs_df["seed"] == int(smoke_row["seed"]))
].copy()
display(smoke_runs[["scenario", "model_label", "seed", "wandb_run_name", "run_dir"]])

smoke_result = clu.evaluate_map_constraint(
    smoke_runs,
    smoke_row["scenario"],
    "default",
    checkpoint_choice=CHECKPOINT_CHOICE,
    device=DEVICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=SMOKE_EVAL_SEEDS,
    eval_seed_batch_size=min(EVAL_SEED_BATCH_SIZE, len(SMOKE_EVAL_SEEDS)),
    duplicate_policy=DUPLICATE_POLICY,
)
display(smoke_result["run_results"])
display(smoke_result["seed_summary"])
display(smoke_result["model_summary"])

## Full evaluation

Each checkpoint is evaluated with 64 individual evaluation seeds and one episode per seed.

In [ ]:
SCENARIO_RESULTS = {}
for scenario in TARGET_SCENARIOS:
    result = clu.evaluate_map(
        runs_df,
        scenario,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
        eval_seeds=EVAL_SEEDS,
        eval_seed_batch_size=EVAL_SEED_BATCH_SIZE,
        model_order=TARGET_MODELS,
        duplicate_policy=DUPLICATE_POLICY,
    )
    SCENARIO_RESULTS[scenario] = result
    print(f"\n=== {scenario} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    latex = result["latex"].replace("Deterministic SMAX win rate", "Deterministic StagHunt success rate")
    latex = latex.replace("\\label{tab:smax_", "\\label{tab:sh_")
    print(latex)

In [ ]:
for scenario, result in SCENARIO_RESULTS.items():
    run_df = result["run_results"].copy()
    seed_summary = result["seed_summary"].copy()
    model_summary = result["model_summary"].copy()

    eval_seed_sets = (
        run_df.groupby(["model_label", "train_seed"])["eval_seed"]
        .agg(lambda values: tuple(sorted(set(int(v) for v in values))))
        .reset_index(name="eval_seed_set")
    )
    assert (eval_seed_sets.groupby("model_label")["eval_seed_set"].nunique() == 1).all()
    assert not (run_df["train_seed"] == run_df["eval_seed"]).any()
    assert (seed_summary["n_runs"] == 1).all() if DUPLICATE_POLICY != "all" else True
    expected_std = seed_summary.groupby("model_label")["eval_success_rate"].std().fillna(0.0)
    observed_std = model_summary.set_index("model_label")["std_success_rate"].fillna(0.0)
    assert (expected_std - observed_std).abs().max() < 1.0e-12

print("Protocol assertions passed.")

In [ ]:
combined_seed_summary = pd.concat(
    [result["seed_summary"] for result in SCENARIO_RESULTS.values()],
    ignore_index=True,
)
combined_model_summary = pd.concat(
    [result["model_summary"] for result in SCENARIO_RESULTS.values()],
    ignore_index=True,
)
display(combined_seed_summary)
display(combined_model_summary)